# 18 · Safety State Machine 与 Degraded Mode

corner-case rule 不应只是一个孤立的二分类器。它需要读取 sensor health、localization covariance、planner disagreement、TTC、时间戳 age 和 ODD 状态，并决定车辆是否继续运行、降速、执行最小风险动作或退出 ODD。

本 notebook 实现一个带有恢复滞回的 safety state machine，并报告：

- fault 到降级状态的检测 latency；
- MINIMAL_RISK 的触发原因；
- nominal 条件下的 false fallback；
- 传感器恢复后重新进入 nominal 的稳定时间。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from ipywidgets import interact, FloatSlider

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.grid"] = True

STATES = ["NOMINAL", "DEGRADED", "MINIMAL_RISK", "ODD_EXIT"]
ACTIONS = {
    "NOMINAL": "learned_policy",
    "DEGRADED": "slow_down_and_monitor",
    "MINIMAL_RISK": "pull_over_or_stop",
    "ODD_EXIT": "stop_accepting_mission",
}


@dataclass
class SafetyStateMachine:
    health_threshold: float = 0.70
    localization_threshold_m: float = 0.80
    ttc_threshold_s: float = 2.50
    recovery_cycles: int = 8
    state: str = "NOMINAL"
    good_cycles: int = 0

    def update(self, sensor_health, localization_sigma, ttc, planner_disagreement, odd_ok, age_s):
        hard_fault = sensor_health < 0.40 or localization_sigma > 1.50 or age_s > 0.60
        immediate_risk = ttc < self.ttc_threshold_s or planner_disagreement > 0.65
        degraded = sensor_health < self.health_threshold or localization_sigma > self.localization_threshold_m or age_s > 0.15
        if not odd_ok:
            self.state = "ODD_EXIT"
        elif hard_fault or immediate_risk:
            self.state = "MINIMAL_RISK"
        elif degraded:
            self.state = "DEGRADED"
            self.good_cycles = 0
        else:
            self.good_cycles += 1
            if self.state in {"DEGRADED", "MINIMAL_RISK"} and self.good_cycles >= self.recovery_cycles:
                self.state = "NOMINAL"
        return self.state


def make_fault_trace(n=600, seed=31):
    rng = np.random.default_rng(seed)
    t = np.arange(n) * 0.1
    health = np.clip(0.96 + rng.normal(0, 0.015, n), 0, 1)
    sigma = np.clip(0.25 + rng.normal(0, 0.03, n), 0, None)
    ttc = np.full(n, 8.0)
    disagreement = np.clip(rng.normal(0.10, 0.03, n), 0, 1)
    age = np.clip(rng.normal(0.04, 0.01, n), 0, None)
    odd_ok = np.ones(n, dtype=bool)
    health[180:250] = 0.58
    sigma[180:250] = 1.00
    age[180:250] = 0.22
    ttc[225:245] = 1.8
    disagreement[225:245] = 0.80
    odd_ok[430:480] = False
    return pd.DataFrame({"time_s": t, "sensor_health": health, "localization_sigma": sigma,
                         "ttc": ttc, "planner_disagreement": disagreement, "age_s": age, "odd_ok": odd_ok})


trace = make_fault_trace()
machine = SafetyStateMachine()
trace["state"] = [machine.update(**row) for row in trace.drop(columns="time_s").to_dict("records")]
trace["action"] = trace["state"].map(ACTIONS)
display(trace["state"].value_counts())


In [ ]:
state_code = {state: i for i, state in enumerate(STATES)}
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(11, 8))
axes[0].plot(trace["time_s"], trace["sensor_health"], label="sensor health")
axes[0].axhline(0.70, color="black", linestyle="--")
axes[0].set_ylabel("health")
axes[1].plot(trace["time_s"], trace["ttc"], label="TTC")
axes[1].axhline(2.5, color="red", linestyle="--")
axes[1].plot(trace["time_s"], trace["localization_sigma"], label="localization sigma")
axes[1].legend()
axes[1].set_ylabel("risk signal")
axes[2].step(trace["time_s"], trace["state"].map(state_code), where="post")
axes[2].set_yticks(list(state_code.values()), list(state_code))
axes[2].set_xlabel("time / s")
axes[2].set_ylabel("state")
plt.show()


In [ ]:
def transition_report(frame):
    transitions = frame["state"].ne(frame["state"].shift()).sum() - 1
    minimal_risk = frame["state"].eq("MINIMAL_RISK")
    nominal_window = frame.iloc[:170]
    return {
        "transition_count": int(transitions),
        "minimal_risk_seconds": float(minimal_risk.sum() * 0.1),
        "nominal_false_fallback_rate": float(nominal_window["state"].ne("NOMINAL").mean()),
        "first_degraded_or_risk_s": float(frame.loc[frame["state"].ne("NOMINAL"), "time_s"].iloc[0]),
    }


print(transition_report(trace))


### 交互练习

1. 调大 `ttc_threshold_s`，观察 safety recall 与 false fallback 的变化；
2. 给状态机增加 `map_version_valid` 和 `compute_budget_ok`；
3. 让 `ODD_EXIT` 只能由系统健康恢复后重置，而不是每帧自动回退；
4. 为每次状态转换记录 `reason`，形成可以审计的 event log；
5. 设计独立于 learned policy 的 rule monitor，并说明它为什么不能复用同一模型的置信度。


In [ ]:
def inspect_threshold(ttc_threshold_s=2.5, recovery_cycles=8):
    machine = SafetyStateMachine(ttc_threshold_s=ttc_threshold_s, recovery_cycles=int(recovery_cycles))
    local = trace.copy()
    local["state"] = [machine.update(**row) for row in local.drop(columns=["time_s", "state", "action"], errors="ignore").to_dict("records")]
    print(transition_report(local))


interact(
    inspect_threshold,
    ttc_threshold_s=FloatSlider(value=2.5, min=1.0, max=5.0, step=0.25),
    recovery_cycles=FloatSlider(value=8, min=1, max=20, step=1),
)


## 完成标准

最终报告必须包含状态转换表、触发原因、检测 latency、恢复 latency 和 nominal false fallback。单独报告一个 rule 的 precision/recall 不足以说明系统安全性。
